**Appendix D – Autodiff**

_This notebook contains toy implementations of various autodiff techniques, to explain how they work._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/extra_autodiff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/extra_autodiff.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

# Introduction

Suppose we want to compute the gradients of the function $f(x,y)=x^2y + y + 2$ with regards to the parameters x and y:

In [ ]:
def f(x,y):
    return x*x*y + y + 2  # 定义函数 f(x,y) = x²y + y + 2，返回计算结果

One approach is to solve this analytically:

$\dfrac{\partial f}{\partial x} = 2xy$

$\dfrac{\partial f}{\partial y} = x^2 + 1$

In [ ]:
def df(x,y):
    return 2*x*y, x*x + 1  # 返回 f 对 x 的偏导数 2xy 和对 y 的偏导数 x²+1

So for example $\dfrac{\partial f}{\partial x}(3,4) = 24$ and $\dfrac{\partial f}{\partial y}(3,4) = 10$.

In [ ]:
df(3, 4)  # 计算 x=3, y=4 时的偏导数，期望结果为 (24, 10)

(24, 10)

Perfect! We can also find the equations for the second order derivatives (also called Hessians):

$\dfrac{\partial^2 f}{\partial x \partial x} = \dfrac{\partial (2xy)}{\partial x} = 2y$

$\dfrac{\partial^2 f}{\partial x \partial y} = \dfrac{\partial (2xy)}{\partial y} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial x} = \dfrac{\partial (x^2 + 1)}{\partial x} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial y} = \dfrac{\partial (x^2 + 1)}{\partial y} = 0$

At x=3 and y=4, these Hessians are respectively 8, 6, 6, 0. Let's use the equations above to compute them:

In [ ]:
def d2f(x, y):
    # 返回二阶偏导数（海森矩阵）
    # 第一行: [∂²f/∂x², ∂²f/∂x∂y] = [2y, 2x]
    # 第二行: [∂²f/∂y∂x, ∂²f/∂y²] = [2x, 0]
    return [2*y, 2*x], [2*x, 0]

In [ ]:
d2f(3, 4)  # 计算 x=3, y=4 时的海森矩阵，期望结果为 ([8, 6], [6, 0])

([8, 6], [6, 0])

Perfect, but this requires some mathematical work. It is not too hard in this case, but for a deep neural network, it is pratically impossible to compute the derivatives this way. So let's look at various ways to automate this!

# Numeric differentiation

Here, we compute an approxiation of the gradients using the equation: $\dfrac{\partial f}{\partial x} = \displaystyle{\lim_{\epsilon \to 0}}\dfrac{f(x+\epsilon, y) - f(x, y)}{\epsilon}$ (and there is a similar definition for $\dfrac{\partial f}{\partial y}$).

In [ ]:
def gradients(func, vars_list, eps=0.0001):
    partial_derivatives = []  # 存储各变量的偏导数
    base_func_eval = func(*vars_list)  # 计算函数在原始点的值
    for idx in range(len(vars_list)):  # 遍历每个变量
        tweaked_vars = vars_list[:]  # 复制变量列表
        tweaked_vars[idx] += eps  # 对当前变量加一个微小扰动 ε
        tweaked_func_eval = func(*tweaked_vars)  # 计算扰动后的函数值
        derivative = (tweaked_func_eval - base_func_eval) / eps  # 用差分法近似偏导数
        partial_derivatives.append(derivative)  # 将偏导数添加到结果列表
    return partial_derivatives  # 返回所有偏导数

In [ ]:
def df(x, y):
    return gradients(f, [x, y])  # 使用数值微分法计算 f 对 x 和 y 的偏导数

In [ ]:
df(3, 4)  # 计算 x=3, y=4 时的近似偏导数，应接近 (24, 10)

[24.000400000048216, 10.000000000047748]

It works well!

The good news is that it is pretty easy to compute the Hessians. First let's create functions that compute the first order partial derivatives (also called Jacobians):

In [ ]:
def dfdx(x, y):
    return gradients(f, [x,y])[0]  # 计算 f 对 x 的偏导数（雅可比矩阵第一项）

def dfdy(x, y):
    return gradients(f, [x,y])[1]  # 计算 f 对 y 的偏导数（雅可比矩阵第二项）

dfdx(3., 4.), dfdy(3., 4.)  # 验证 x=3, y=4 时的偏导数

(24.000400000048216, 10.000000000047748)

Now we can simply apply the `gradients()` function to these functions:

In [ ]:
def d2f(x, y):
    # 对一阶偏导数函数再次求数值梯度，得到二阶偏导数（海森矩阵）
    return [gradients(dfdx, [x, y]), gradients(dfdy, [x, y])]

In [ ]:
d2f(3, 4)  # 计算 x=3, y=4 时的近似海森矩阵

[[7.999999951380232, 6.000099261882497],
 [6.000099261882497, -1.4210854715202004e-06]]

So everything works well, but the result is approximate, and computing the gradients of a function with regards to $n$ variables requires calling that function $n$ times. In deep neural nets, there are often thousands of parameters to tweak using gradient descent (which requires computing the gradients of the loss function with regards to each of these parameters), so this approach would be much too slow.

## Implementing a Toy Computation Graph

Rather than this numerical approach, let's implement some symbolic autodiff techniques. For this, we will need to define classes to represent constants, variables and operations.

In [ ]:
# 常量节点类
class Const(object):
    def __init__(self, value):
        self.value = value  # 存储常量值
    def evaluate(self):
        return self.value  # 求值时直接返回常量值
    def __str__(self):
        return str(self.value)  # 字符串表示为常量值

# 变量节点类
class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value  # 变量的当前值
        self.name = name  # 变量名称
    def evaluate(self):
        return self.value  # 求值时返回变量当前值
    def __str__(self):
        return self.name  # 字符串表示为变量名

# 二元运算符基类
class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a  # 左操作数
        self.b = b  # 右操作数

# 加法运算节点
class Add(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() + self.b.evaluate()  # 递归求值：左操作数 + 右操作数
    def __str__(self):
        return "{} + {}".format(self.a, self.b)  # 字符串表示为 "a + b"

# 乘法运算节点
class Mul(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() * self.b.evaluate()  # 递归求值：左操作数 × 右操作数
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)  # 字符串表示为 "(a) * (b)"

Good, now we can build a computation graph to represent the function $f$:

In [ ]:
x = Var("x")  # 创建变量 x
y = Var("y")  # 创建变量 y
f = Add(Mul(Mul(x, x), y), Add(y, Const(2)))  # 构建计算图: f(x,y) = x²y + y + 2

And we can run this graph to compute $f$ at any point, for example $f(3, 4)$.

In [ ]:
x.value = 3  # 设置 x = 3
y.value = 4  # 设置 y = 4
f.evaluate()  # 计算 f(3,4) = 9×4 + 4 + 2 = 42

42

Perfect, it found the ultimate answer.

## Computing gradients

The autodiff methods we will present below are all based on the *chain rule*.

Suppose we have two functions $u$ and $v$, and we apply them sequentially to some input $x$, and we get the result $z$. So we have $z = v(u(x))$, which we can rewrite as $z = v(s)$ and $s = u(x)$. Now we can apply the chain rule to get the partial derivative of the output $z$ with regards to the input $x$:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s}{\partial x} \cdot \dfrac{\partial z}{\partial s}$

Now if $z$ is the output of a sequence of functions which have intermediate outputs $s_1, s_2, ..., s_n$, the chain rule still applies:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s_1}{\partial x} \cdot \dfrac{\partial s_2}{\partial s_1} \cdot \dfrac{\partial s_3}{\partial s_2} \cdot \dots \cdot \dfrac{\partial s_{n-1}}{\partial s_{n-2}} \cdot \dfrac{\partial s_n}{\partial s_{n-1}} \cdot \dfrac{\partial z}{\partial s_n}$

In forward mode autodiff, the algorithm computes these terms "forward" (i.e., in the same order as the computations required to compute the output $z$), that is from left to right: first $\dfrac{\partial s_1}{\partial x}$, then $\dfrac{\partial s_2}{\partial s_1}$, and so on. In reverse mode autodiff, the algorithm computes these terms "backwards", from right to left: first $\dfrac{\partial z}{\partial s_n}$, then $\dfrac{\partial s_n}{\partial s_{n-1}}$, and so on.

For example, suppose you want to compute the derivative of the function $z(x)=\sin(x^2)$ at x=3, using forward mode autodiff. The algorithm would first compute the partial derivative $\dfrac{\partial s_1}{\partial x}=\dfrac{\partial x^2}{\partial x}=2x=6$. Next, it would compute $\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1}= 6 \cdot \dfrac{\partial \sin(s_1)}{\partial s_1}=6 \cdot \cos(s_1)=6 \cdot \cos(3^2)\approx-5.46$.

Let's verify this result using the `gradients()` function defined earlier:

In [ ]:
from math import sin  # 导入 sin 函数

def z(x):
    return sin(x**2)  # 定义函数 z(x) = sin(x²)

gradients(z, [3])  # 用数值微分法计算 z 在 x=3 处的导数，应约为 -5.46

[-5.46761419430053]

Look good. Now let's do the same thing using reverse mode autodiff. This time the algorithm would start from the right hand side so it would compute $\dfrac{\partial z}{\partial s_1} = \dfrac{\partial \sin(s_1)}{\partial s_1}=\cos(s_1)=\cos(3^2)\approx -0.91$. Next it would compute $\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1} \approx \dfrac{\partial s_1}{\partial x} \cdot -0.91 = \dfrac{\partial x^2}{\partial x} \cdot -0.91=2x \cdot -0.91 = 6\cdot-0.91=-5.46$.

Of course both approaches give the same result (except for rounding errors), and with a single input and output they involve the same number of computations. But when there are several inputs or outputs, they can have very different performance. Indeed, if there are many inputs, the right-most terms will be needed to compute the partial derivatives with regards to each input, so it is a good idea to compute these right-most terms first. That means using reverse-mode autodiff. This way, the right-most terms can be computed just once and used to compute all the partial derivatives. Conversely, if there are many outputs, forward-mode is generally preferable because the left-most terms can be computed just once to compute the partial derivatives of the different outputs. In Deep Learning, there are typically thousands of model parameters, meaning there are lots of inputs, but few outputs. In fact, there is generally just one output during training: the loss. This is why reverse mode autodiff is used in TensorFlow and all major Deep Learning libraries.

There's one additional complexity in reverse mode autodiff: the value of $s_i$ is generally required when computing $\dfrac{\partial s_{i+1}}{\partial s_i}$, and computing $s_i$ requires first computing $s_{i-1}$, which requires computing $s_{i-2}$, and so on. So basically, a first pass forward through the network is required to compute $s_1$, $s_2$, $s_3$, $\dots$, $s_{n-1}$ and $s_n$, and then the algorithm can compute the partial derivatives from right to left. Storing all the intermediate values $s_i$ in RAM is sometimes a problem, especially when handling images, and when using GPUs which often have limited RAM: to limit this problem, one can reduce the number of layers in the neural network, or configure TensorFlow to make it swap these values from GPU RAM to CPU RAM. Another approach is to only cache every other intermediate value, $s_1$, $s_3$, $s_5$, $\dots$, $s_{n-4}$, $s_{n-2}$ and $s_n$. This means that when the algorithm computes the partial derivatives, if an intermediate value $s_i$ is missing, it will need to recompute it based on the previous intermediate value $s_{i-1}$. This trades off CPU for RAM (if you are interested, check out [this paper](https://pdfs.semanticscholar.org/f61e/9fd5a4878e1493f7a6b03774a61c17b7e9a4.pdf)).

### Forward mode autodiff

In [ ]:
# 为常量定义梯度方法：常量对任何变量的导数为 0
Const.gradient = lambda self, var: Const(0)
# 为变量定义梯度方法：变量对自身的导数为 1，对其他变量的导数为 0
Var.gradient = lambda self, var: Const(1) if self is var else Const(0)
# 加法的梯度规则：d(a+b)/dvar = da/dvar + db/dvar
Add.gradient = lambda self, var: Add(self.a.gradient(var), self.b.gradient(var))
# 乘法的梯度规则（乘法法则）：d(a*b)/dvar = a * db/dvar + da/dvar * b
Mul.gradient = lambda self, var: Add(Mul(self.a, self.b.gradient(var)), Mul(self.a.gradient(var), self.b))

x = Var(name="x", init_value=3.)  # 创建变量 x，初始值为 3.0
y = Var(name="y", init_value=4.)  # 创建变量 y，初始值为 4.0
f = Add(Mul(Mul(x, x), y), Add(y, Const(2)))  # 构建计算图: f(x,y) = x²y + y + 2

dfdx = f.gradient(x)  # 符号化计算 ∂f/∂x = 2xy
dfdy = f.gradient(y)  # 符号化计算 ∂f/∂y = x² + 1

In [ ]:
dfdx.evaluate(), dfdy.evaluate()  # 求值一阶偏导数，期望结果为 (24.0, 10.0)

(24.0, 10.0)

Since the output of the `gradient()` method is fully symbolic, we are not limited to the first order derivatives, we can also compute second order derivatives, and so on:

In [ ]:
d2fdxdx = dfdx.gradient(x)  # 符号化计算 ∂²f/∂x² = 2y
d2fdxdy = dfdx.gradient(y)  # 符号化计算 ∂²f/∂x∂y = 2x
d2fdydx = dfdy.gradient(x)  # 符号化计算 ∂²f/∂y∂x = 2x
d2fdydy = dfdy.gradient(y)  # 符号化计算 ∂²f/∂y² = 0

In [ ]:
# 求值二阶偏导数（海森矩阵），期望结果为 [[8, 6], [6, 0]]
[[d2fdxdx.evaluate(), d2fdxdy.evaluate()],
 [d2fdydx.evaluate(), d2fdydy.evaluate()]]

[[8.0, 6.0], [6.0, 0.0]]

Note that the result is now exact, not an approximation (up to the limit of the machine's float precision, of course).

### Forward mode autodiff using dual numbers

A nice way to apply forward mode autodiff is to use [dual numbers](https://en.wikipedia.org/wiki/Dual_number). In short, a dual number $z$ has the form $z = a + b\epsilon$, where $a$ and $b$ are real numbers, and $\epsilon$ is an infinitesimal number, positive but smaller than all real numbers, and such that $\epsilon^2=0$.
It can be shown that $f(x + \epsilon) = f(x) + \dfrac{\partial f}{\partial x}\epsilon$, so simply by computing $f(x + \epsilon)$ we get both the value of $f(x)$ and the partial derivative of $f$ with regards to $x$. 

Dual numbers have their own arithmetic rules, which are generally quite natural. For example:

**Addition**

$(a_1 + b_1\epsilon) + (a_2 + b_2\epsilon) = (a_1 + a_2) + (b_1 + b_2)\epsilon$

**Subtraction**

$(a_1 + b_1\epsilon) - (a_2 + b_2\epsilon) = (a_1 - a_2) + (b_1 - b_2)\epsilon$

**Multiplication**

$(a_1 + b_1\epsilon) \times (a_2 + b_2\epsilon) = (a_1 a_2) + (a_1 b_2 + a_2 b_1)\epsilon + b_1 b_2\epsilon^2 = (a_1 a_2) + (a_1b_2 + a_2b_1)\epsilon$

**Division**

$\dfrac{a_1 + b_1\epsilon}{a_2 + b_2\epsilon} = \dfrac{a_1 + b_1\epsilon}{a_2 + b_2\epsilon} \cdot \dfrac{a_2 - b_2\epsilon}{a_2 - b_2\epsilon} = \dfrac{a_1 a_2 + (b_1 a_2 - a_1 b_2)\epsilon - b_1 b_2\epsilon^2}{{a_2}^2 + (a_2 b_2 - a_2 b_2)\epsilon - {b_2}^2\epsilon} = \dfrac{a_1}{a_2} + \dfrac{a_1 b_2 - b_1 a_2}{{a_2}^2}\epsilon$

**Power**

$(a + b\epsilon)^n = a^n + (n a^{n-1}b)\epsilon$

etc.

Let's create a class to represent dual numbers, and implement a few operations (addition and multiplication). You can try adding some more if you want.

In [ ]:
# 对偶数类，用于前向模式自动微分
class DualNumber(object):
    def __init__(self, value=0.0, eps=0.0):
        self.value = value  # 实部（函数值）
        self.eps = eps  # 对偶部（导数值）
    def __add__(self, b):
        # 加法规则: (a1 + b1ε) + (a2 + b2ε) = (a1+a2) + (b1+b2)ε
        return DualNumber(self.value + self.to_dual(b).value,
                          self.eps + self.to_dual(b).eps)
    def __radd__(self, a):
        return self.to_dual(a).__add__(self)  # 反向加法，处理 普通数 + 对偶数 的情况
    def __mul__(self, b):
        # 乘法规则: (a1 + b1ε) × (a2 + b2ε) = a1*a2 + (a1*b2 + a2*b1)ε
        return DualNumber(self.value * self.to_dual(b).value,
                          self.eps * self.to_dual(b).value + self.value * self.to_dual(b).eps)
    def __rmul__(self, a):
        return self.to_dual(a).__mul__(self)  # 反向乘法，处理 普通数 × 对偶数 的情况
    def __str__(self):
        if self.eps:
            return "{:.1f} + {:.1f}ε".format(self.value, self.eps)  # 有对偶部时显示完整形式
        else:
            return "{:.1f}".format(self.value)  # 无对偶部时只显示实部
    def __repr__(self):
        return str(self)  # repr 与 str 相同
    @classmethod
    def to_dual(cls, n):
        if hasattr(n, "value"):
            return n  # 已经是对偶数，直接返回
        else:
            return cls(n)  # 将普通数转换为对偶数（对偶部为 0）

$3 + (3 + 4 \epsilon) = 6 + 4\epsilon$

In [ ]:
3 + DualNumber(3, 4)  # 验证加法: 3 + (3 + 4ε) = 6 + 4ε

6.0 + 4.0ε

$(3 + 4ε)\times(5 + 7ε)$ = $3 \times 5 + 3 \times 7ε + 4ε \times 5 + 4ε \times 7ε$ = $15 + 21ε + 20ε + 28ε^2$ = $15 + 41ε + 28 \times 0$ = $15 + 41ε$

In [ ]:
DualNumber(3, 4) * DualNumber(5, 7)  # 验证乘法: (3+4ε)×(5+7ε) = 15 + 41ε

15.0 + 41.0ε

Now let's see if the dual numbers work with our toy computation framework:

In [ ]:
x.value = DualNumber(3.0)  # 将 x 设为对偶数 3.0（无对偶部）
y.value = DualNumber(4.0)  # 将 y 设为对偶数 4.0（无对偶部）

f.evaluate()  # 计算 f(3,4)，验证对偶数在计算图中能正常工作

42.0

Yep, sure works. Now let's use this to compute the partial derivatives of $f$ with regards to $x$ and $y$ at x=3 and y=4:

In [ ]:
# 计算 ∂f/∂x：将 x 设为 3+ε，y 设为 4，f 的对偶部即为 ∂f/∂x
x.value = DualNumber(3.0, 1.0)  # x = 3 + ε（对 x 方向加扰动）
y.value = DualNumber(4.0)       # y = 4（无扰动）

dfdx = f.evaluate().eps  # 提取对偶部，即 ∂f/∂x 在 (3,4) 处的值

# 计算 ∂f/∂y：将 x 设为 3，y 设为 4+ε，f 的对偶部即为 ∂f/∂y
x.value = DualNumber(3.0)       # x = 3（无扰动）
y.value = DualNumber(4.0, 1.0)  # y = 4 + ε（对 y 方向加扰动）

dfdy = f.evaluate().eps  # 提取对偶部，即 ∂f/∂y 在 (3,4) 处的值

In [ ]:
dfdx  # 输出 ∂f/∂x = 24.0

24.0

In [ ]:
dfdy  # 输出 ∂f/∂y = 10.0

10.0

Great! However, in this implementation we are limited to first order derivatives.
Now let's look at reverse mode.

### Reverse mode autodiff

Let's rewrite our toy framework to add reverse mode autodiff:

In [ ]:
# 常量节点类（支持反向传播）
class Const(object):
    def __init__(self, value):
        self.value = value  # 存储常量值
    def evaluate(self):
        return self.value  # 前向传播：返回常量值
    def backpropagate(self, gradient):
        pass  # 反向传播：常量无需更新梯度，直接忽略
    def __str__(self):
        return str(self.value)  # 字符串表示

# 变量节点类（支持反向传播）
class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value  # 变量的当前值
        self.name = name  # 变量名称
        self.gradient = 0  # 累积梯度，初始为 0
    def evaluate(self):
        return self.value  # 前向传播：返回变量值
    def backpropagate(self, gradient):
        self.gradient += gradient  # 反向传播：累加来自上游的梯度
    def __str__(self):
        return self.name  # 字符串表示

# 二元运算符基类
class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a  # 左操作数
        self.b = b  # 右操作数

# 加法运算节点（支持反向传播）
class Add(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() + self.b.evaluate()  # 前向传播：计算 a + b
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient)  # 加法的梯度直接传递给 a
        self.b.backpropagate(gradient)  # 加法的梯度直接传递给 b
    def __str__(self):
        return "{} + {}".format(self.a, self.b)

# 乘法运算节点（支持反向传播）
class Mul(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() * self.b.evaluate()  # 前向传播：计算 a × b
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient * self.b.value)  # 对 a 的梯度 = 上游梯度 × b 的值
        self.b.backpropagate(gradient * self.a.value)  # 对 b 的梯度 = 上游梯度 × a 的值
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)

In [ ]:
x = Var("x", init_value=3)  # 创建变量 x，初始值为 3
y = Var("y", init_value=4)  # 创建变量 y，初始值为 4
f = Add(Mul(Mul(x, x), y), Add(y, Const(2)))  # 构建计算图: f(x,y) = x²y + y + 2

result = f.evaluate()  # 前向传播：计算 f(3,4) 的值
f.backpropagate(1.0)  # 反向传播：从输出端传入梯度 1.0，计算各变量的梯度

In [ ]:
print(f)  # 打印计算图的符号表达式

((x) * (x)) * (y) + y + 2


In [ ]:
result  # 输出 f(3,4) 的计算结果 = 42

42

In [ ]:
x.gradient  # 输出 ∂f/∂x = 24（通过反向传播累积得到）

24.0

In [ ]:
y.gradient  # 输出 ∂f/∂y = 10（通过反向传播累积得到）

10.0

Again, in this implementation the outputs are just numbers, not symbolic expressions, so we are limited to first order derivatives. However, we could have made the `backpropagate()` methods return symbolic expressions rather than values (e.g., return `Add(2,3)` rather than 5). This would make it possible to compute second order gradients (and beyond). This is what TensorFlow does, as do all the major libraries that implement autodiff.

### Reverse mode autodiff using TensorFlow

In [ ]:
import tensorflow as tf  # 导入 TensorFlow 库

In [ ]:
x = tf.Variable(3.)  # 创建 TensorFlow 变量 x = 3.0
y = tf.Variable(4.)  # 创建 TensorFlow 变量 y = 4.0

with tf.GradientTape() as tape:  # 使用 GradientTape 记录计算过程以便自动微分
    f = x*x*y + y + 2  # 在 tape 上下文中计算 f(x,y) = x²y + y + 2

jacobians = tape.gradient(f, [x, y])  # 计算 f 对 [x, y] 的一阶偏导数（雅可比）
jacobians  # 输出 [24.0, 10.0]

[<tf.Tensor: shape=(), dtype=float32, numpy=24.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]

Since everything is symbolic, we can compute second order derivatives, and beyond:

In [ ]:
x = tf.Variable(3.)  # 创建 TensorFlow 变量 x = 3.0
y = tf.Variable(4.)  # 创建 TensorFlow 变量 y = 4.0

# persistent=True 允许多次调用 tape.gradient()（默认只能调用一次）
with tf.GradientTape(persistent=True) as tape:
    f = x*x*y + y + 2  # 计算 f(x,y) = x²y + y + 2
    df_dx, df_dy = tape.gradient(f, [x, y])  # 计算一阶偏导数: ∂f/∂x, ∂f/∂y

# 对一阶偏导数再次求导，得到二阶偏导数（海森矩阵）
d2f_d2x, d2f_dydx = tape.gradient(df_dx, [x, y])  # ∂²f/∂x² 和 ∂²f/∂y∂x
d2f_dxdy, d2f_d2y = tape.gradient(df_dy, [x, y])  # ∂²f/∂x∂y 和 ∂²f/∂y²
del tape  # 手动删除 persistent tape 以释放资源

hessians = [[d2f_d2x, d2f_dydx], [d2f_dxdy, d2f_d2y]]  # 组织为海森矩阵
hessians  # 输出 [[8.0, 6.0], [6.0, None]]（None 表示 ∂²f/∂y²=0）

[[<tf.Tensor: shape=(), dtype=float32, numpy=8.0>,
  <tf.Tensor: shape=(), dtype=float32, numpy=6.0>],
 [<tf.Tensor: shape=(), dtype=float32, numpy=6.0>, None]]

Note that when we compute the derivative of a tensor with regards to a variable that it does not depend on, instead of returning 0.0, the `gradient()` function returns `None`.

And that's all folks! Hope you enjoyed this notebook.